In [3]:
import torch
import GPUtil
import os

GPUtil.showUtilization()

if torch.cuda.is_available():
  print("GPU is available")
else:
  print("GPU is not available")

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

| ID | GPU | MEM |
------------------
|  0 |  0% |  0% |
GPU is available


In [4]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, LlamaTokenizer
from huggingface_hub import notebook_login
from datasets import load_dataset
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

if "COLAB_GPU" in os.environ:
  from google.colab import output
  output.enable_custom_widget_manager()


In [5]:
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()

hf_token = os.getenv("HF_TOKEN")
login(hf_token)

In [6]:
base_model_id="meta-llama/Llama-2-7b-chat-hf"

bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model=AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [7]:
!git clone https://github.com/poloclub/Fine-tuning-LLMs.git

fatal: destination path 'Fine-tuning-LLMs' already exists and is not an empty directory.


In [8]:
train_dataset=load_dataset("text", data_files={"train":
                                               ["/content/Fine-tuning-LLMs/data/hawaii_wf_4.txt", "/content/Fine-tuning-LLMs/data/hawaii_wf_2.txt"]}, split="train")

In [9]:
train_dataset["text"][0]

'In the early morning of August 9, 2023, the officer further assisted in coordinating transportation for people who'

In [10]:
tokenizer=LlamaTokenizer.from_pretrained(base_model_id, use_fast=False, trust_remote_code=True, add_eos_toekn=True)

if tokenizer.pad_token is None:
  tokenizer.add_special_tokens({"pad_token":tokenizer.eos_token})

In [11]:
tokenized_train_dataset=[]
for phrase in train_dataset:
  tokenized_train_dataset.append(tokenizer(phrase["text"]))

In [12]:
tokenized_train_dataset[1]

{'input_ids': [1, 750, 4586, 25447, 297, 278, 23474, 304, 10169, 278, 3974, 29892, 5662, 3864, 896, 7450, 278, 11176, 14703, 27709, 23511, 29889], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [13]:
tokenizer.eos_token

'</s>'

In [14]:
model.gradient_checkpointing_enable()
model=prepare_model_for_kbit_training(model)

config=LoraConfig(
    r=8,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model=get_peft_model(model, config)


In [15]:
trainer=transformers.Trainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    args=transformers.TrainingArguments(
        output_dir="./finetuned_Model",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=1e-4,
        max_steps=61, # Changed from 20 to 61 to ensure at least one epoch is completed
        bf16=False,
        optim="paged_adamw_8bit",
        logging_dir="./log",
        save_strategy="epoch",
        save_steps=50,
        logging_steps=10
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

model.config.use_cache=False
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,3.491837
20,3.138632
30,3.020218
40,2.819741
50,2.733221
60,2.611103


TrainOutput(global_step=61, training_loss=2.9344200560303983, metrics={'train_runtime': 311.1943, 'train_samples_per_second': 0.784, 'train_steps_per_second': 0.196, 'total_flos': 253177340731392.0, 'train_loss': 2.9344200560303983, 'epoch': 1.0})

In [16]:
# output_dir = "./fine_tuned_llama2_model"

# trainer.model.save_pretrained(output_dir)
# tokenizer.save_pretrained(output_dir)

# print(f"Model and tokenizer saved to {output_dir}")

In [17]:
num_training_samples = len(tokenized_train_dataset)
print(f"Number of training samples: {num_training_samples}")

per_device_train_batch_size = trainer.args.per_device_train_batch_size
gradient_accumulation_steps = trainer.args.gradient_accumulation_steps

effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps
print(f"Effective batch size: {effective_batch_size}")

steps_per_epoch = num_training_samples / effective_batch_size
print(f"Steps per epoch: {steps_per_epoch}")

Number of training samples: 241
Effective batch size: 4
Steps per epoch: 60.25


In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig, LlamaTokenizer
from peft import PeftModel

base_model_id="meta-llama/Llama-2-7b-chat-hf"

nf4Config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer=LlamaTokenizer.from_pretrained(base_model_id, use_fast=False, trust_remote_code=True, add_eos_token=True)

base_model=AutoModelForCausalLM.from_pretrained(base_model_id,
                                                quantization_config=nf4Config,
                                                device_map="auto",
                                                trust_remote_code=True,
                                                token=True)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 86.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 65.81 MiB is free. Including non-PyTorch memory, this process has 14.50 GiB memory in use. Of the allocated memory 14.04 GiB is allocated by PyTorch, and 334.23 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
tokenizer=LlamaTokenizer.from_pretrained(base_model_id, use_fast=False, trust_remote_code=True, add_eos_token=True)

modelFinetuned=PeftModel.from_pretrained(base_model, "finetunedModel/checkpoint-20")

In [ ]:
user_question="When did Hawaii wildfires start?"

eval_prompt=f"Question {user_question} Just answer this question acurately and concisely"

prompTokenized=tokenizer(eval_prompt, return_tensors="pt").to("cuda")

modelFinetuned.eval()

with torch.no_grad():
  print(tokenizer.decode(modelFinetuned.generate(**prompTokenized, max_new_tokens=1024)[0], skip_special_tokens=True))
  torch.cuda.empty_cache()